# Database Design & Normalisation

## Objectives

- Identify database entities.
- Analyse relationships between entities.
- Identify primary and foreign keys.
- Validate functional dependencies.
- Design a relational database.
- Prepare data for PostgreSQL.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/inventory_cleaned.csv")

df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,...,inventory_value,stock_status,reorder_quantity,reorder_required,year,month,month_name,day_of_week,quarter,is_weekend
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,7738.50,In Stock,0,False,2022,1,January,Saturday,1,True
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,12854.04,In Stock,0,False,2022,1,January,Saturday,1,True
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,2854.98,In Stock,0,False,2022,1,January,Saturday,1,True
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,15345.68,Overstock,0,False,2022,1,January,Saturday,1,True
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,12224.24,In Stock,0,False,2022,1,January,Saturday,1,True


In [2]:
df.columns.tolist()

['date',
 'store_id',
 'product_id',
 'category',
 'region',
 'inventory_level',
 'units_sold',
 'units_ordered',
 'demand_forecast',
 'price',
 'discount',
 'weather_condition',
 'holiday_promotion',
 'competitor_pricing',
 'seasonality',
 'discounted_price',
 'revenue',
 'inventory_value',
 'stock_status',
 'reorder_quantity',
 'reorder_required',
 'year',
 'month',
 'month_name',
 'day_of_week',
 'quarter',
 'is_weekend']

In [3]:
print("Total records:", len(df))
print("Unique products:", df["product_id"].nunique())
print("Unique stores:", df["store_id"].nunique())
print("Unique categories:", df["category"].nunique())
print("Unique regions:", df["region"].nunique())

Total records: 73100
Unique products: 20
Unique stores: 5
Unique categories: 5
Unique regions: 4


In [4]:
df.groupby("product_id")["category"].nunique().value_counts()

category
5    20
Name: count, dtype: int64

In [6]:
df.groupby("store_id")["region"].nunique().value_counts()

region
4    5
Name: count, dtype: int64

In [7]:
product_conflicts = (
    df.groupby("product_id")["category"].nunique() > 1
).sum()

store_conflicts = (
    df.groupby("store_id")["region"].nunique() > 1
).sum()

print("Product category conflicts:", product_conflicts)
print("Store region conflicts:", store_conflicts)

Product category conflicts: 20
Store region conflicts: 5


## Data Consistency Investigation

Product Category Analysis

In [8]:
df.groupby("product_id")["category"].unique()

product_id
P0001    [Groceries, Toys, Electronics, Clothing, Furni...
P0002    [Toys, Clothing, Electronics, Furniture, Groce...
P0003    [Toys, Furniture, Groceries, Clothing, Electro...
P0004    [Toys, Groceries, Furniture, Electronics, Clot...
P0005    [Electronics, Toys, Clothing, Groceries, Furni...
P0006    [Groceries, Furniture, Electronics, Toys, Clot...
P0007    [Furniture, Groceries, Electronics, Clothing, ...
P0008    [Clothing, Electronics, Toys, Furniture, Groce...
P0009    [Electronics, Clothing, Groceries, Furniture, ...
P0010    [Toys, Electronics, Groceries, Clothing, Furni...
P0011    [Furniture, Groceries, Toys, Electronics, Clot...
P0012    [Clothing, Groceries, Electronics, Furniture, ...
P0013    [Toys, Clothing, Furniture, Electronics, Groce...
P0014    [Clothing, Electronics, Furniture, Toys, Groce...
P0015    [Clothing, Toys, Furniture, Electronics, Groce...
P0016    [Electronics, Toys, Groceries, Clothing, Furni...
P0017    [Toys, Groceries, Clothing, Electron

Store Region Analysis

In [9]:
df.groupby("store_id")["region"].unique()

store_id
S001    [North, South, West, East]
S002    [South, West, East, North]
S003    [South, West, North, East]
S004    [South, North, West, East]
S005    [West, North, South, East]
Name: region, dtype: object

Product Category Distribution

In [10]:
pd.crosstab(df["product_id"], df["category"])

category,Clothing,Electronics,Furniture,Groceries,Toys
product_id,,,,,
P0001,744,700,720,763,728
P0002,718,727,753,729,728
P0003,717,689,735,759,755
P0004,711,747,725,727,745
P0005,697,743,740,754,721
P0006,731,720,702,777,725
P0007,740,762,736,696,721
P0008,755,758,733,681,728
P0009,766,675,770,721,723


Store Region Distribution

In [11]:
pd.crosstab(df["store_id"], df["region"])

region,East,North,South,West
store_id,,,,
S001,3675,3637,3663,3645
S002,3749,3622,3671,3578
S003,3636,3669,3678,3637
S004,3620,3662,3621,3717
S005,3669,3638,3664,3649


Composite Key Validation

In [12]:
duplicates = df.duplicated(
    subset=["date", "store_id", "product_id"]
).sum()

print("Duplicate composite keys:", duplicates)

Duplicate composite keys: 0


In [13]:
print("Total products:", df["product_id"].nunique())
print("Total stores:", df["store_id"].nunique())

print("Product category conflicts:", product_conflicts)
print("Store region conflicts:", store_conflicts)

Total products: 20
Total stores: 5
Product category conflicts: 20
Store region conflicts: 5


In [14]:
print("Unique product-category combinations:",
      df[["product_id", "category"]].drop_duplicates().shape[0])

print("Unique store-region combinations:",
      df[["store_id", "region"]].drop_duplicates().shape[0])

Unique product-category combinations: 100
Unique store-region combinations: 20


In [15]:
print("Duplicate composite keys:",
      df.duplicated(
          subset=["date", "store_id", "product_id"]
      ).sum())

Duplicate composite keys: 0


## Revised Database Architecture

The data quality investigation identified inconsistent
product-category and store-region relationships.

To preserve the original data, a star schema architecture
was selected, with daily inventory records as the central
fact table and four dimension tables.

Products

In [16]:
products = df[["product_id"]].drop_duplicates()

products = products.sort_values(
    "product_id"
).reset_index(drop=True)

products

,product_id
0,P0001
1,P0002
2,P0003
3,P0004
4,P0005
5,P0006
6,P0007
7,P0008
8,P0009
9,P0010


Stores

In [17]:
stores = df[["store_id"]].drop_duplicates()

stores = stores.sort_values(
    "store_id"
).reset_index(drop=True)

stores

,store_id
0,S001
1,S002
2,S003
3,S004
4,S005


Categories

In [18]:
categories = df[["category"]].drop_duplicates()

categories = categories.sort_values(
    "category"
).reset_index(drop=True)

categories["category_id"] = categories.index + 1

categories = categories[["category_id", "category"]]

categories

,category_id,category
0,1,Clothing
1,2,Electronics
2,3,Furniture
3,4,Groceries
4,5,Toys


Regions

In [19]:
regions = df[["region"]].drop_duplicates()

regions = regions.sort_values(
    "region"
).reset_index(drop=True)

regions["region_id"] = regions.index + 1

regions = regions[["region_id", "region"]]

regions

,region_id,region
0,1,East
1,2,North
2,3,South
3,4,West


In [20]:
daily_inventory = df[[
    "date",
    "product_id",
    "store_id",
    "category",
    "region",
    "inventory_level",
    "units_sold",
    "units_ordered",
    "demand_forecast",
    "price",
    "discount",
    "weather_condition",
    "holiday_promotion",
    "competitor_pricing",
    "seasonality"
]].copy()

daily_inventory.head()

,date,product_id,store_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality
0,2022-01-01,P0001,S001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,P0002,S001,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,P0003,S001,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,P0004,S001,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,P0005,S001,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [21]:
daily_inventory = daily_inventory.merge(
    categories,
    on="category",
    validate="many_to_one"
)

daily_inventory = daily_inventory.merge(
    regions,
    on="region",
    validate="many_to_one"
)

daily_inventory = daily_inventory.drop(
    columns=["category", "region"]
)

daily_inventory.head()

,date,product_id,store_id,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,category_id,region_id
0,2022-01-01,P0001,S001,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,4,2
1,2022-01-01,P0002,S001,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,5,3
2,2022-01-01,P0003,S001,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,5,4
3,2022-01-01,P0004,S001,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,5,2
4,2022-01-01,P0005,S001,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2,1


In [22]:
print("Products:", len(products))
print("Stores:", len(stores))
print("Categories:", len(categories))
print("Regions:", len(regions))
print("Daily inventory records:", len(daily_inventory))

print("Original records:", len(df))
print("Missing values:", daily_inventory.isnull().sum().sum())

assert len(daily_inventory) == len(df)
assert not daily_inventory.duplicated(
    subset=["date", "product_id", "store_id"]
).any()

assert not daily_inventory.isnull().any().any()

print("Database validation passed!")

Products: 20
Stores: 5
Categories: 5
Regions: 4
Daily inventory records: 73100
Original records: 73100
Missing values: 0
Database validation passed!
